In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

if os.environ["OPENAI_API_KEY"] is None:
    raise ValueError("OPENAI_API_KEY environment variable not set.")
else:
    print("OPENAI_API_KEY environment variable is set.")


OPENAI_API_KEY environment variable is set.


In [6]:
!pip3 install -U "autogen-agentchat"
!pip3 install "autogen-ext[openai]"

  Using cached autogen_core-0.7.5-py3-none-any.whl.metadata (2.3 kB)
Using cached autogen_core-0.7.5-py3-none-any.whl (101 kB)
  Attempting uninstall: autogen-core
    Found existing installation: autogen-core 0.7.1
    Uninstalling autogen-core-0.7.1:
      Successfully uninstalled autogen-core-0.7.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogen-ext 0.7.1 requires autogen-core==0.7.1, but you have autogen-core 0.7.5 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Using cached autogen_core-0.7.1-py3-none-any.whl.metadata (2.3 kB)
Using cached autogen_core-0.7.1-py3-none-any.whl (101 kB)
  Attempting uninstall: autogen-core
    Found existing installation: autogen-core 0.7.5
    Uninstalling autogen-core-0.7.5:
      Successfully uninstalled autogen-core-0.7.5
ERROR

In [3]:
# Verify installation
import autogen_agentchat
import autogen_ext
print(f"autogen-agentchat : {autogen_agentchat.__version__}")
print(f"autogen-ext       : {autogen_ext.__version__}")
print("✅ All packages installed correctly")

autogen-agentchat : 0.7.5
autogen-ext       : 0.7.1
✅ All packages installed correctly


## Create simple Agent with Autogen

In [2]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import StructuredMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient


async def web_search(query: str) -> str:
    """Find information on the web"""
    return "AutoGen is a programming framework for building multi-agent applications."


# Create an agent that uses the OpenAI GPT-4o model.
model_client = OpenAIChatCompletionClient(
    model="gpt-4.1-nano",
    # api_key="YOUR_API_KEY",
)


agent = AssistantAgent(name = "SimpleAssitant",model_client=model_client, tools=[web_search],system_message="You are a helpful assistant that can answer questions about AutoGen.")



In [8]:
result = await agent.run(task="Find information on AutoGen")
print(result.messages[-1].content)

AutoGen is a framework that enables the development of multi-agent applications, allowing multiple autonomous agents to work together, communicate, and coordinate to achieve complex goals. It is useful for creating systems that require collaboration, decision-making, and problem-solving across multiple entities.

If you want more specific details or have particular questions about AutoGen, please let me know!


## Userproxy Agent

In [ ]:


Two Agents ->. Planner. Reviewer


Planner will be the AssistantAgent

Reviwer agent will review the plan from the above agent either you can review it with LLM or with User.UserWarning


LLM you use AssistantAgent
Human you use UserProxyAgent

In [9]:
import asyncio
from autogen_core import CancellationToken
from autogen_agentchat.agents import UserProxyAgent
from autogen_agentchat.messages import TextMessage


async def simple_user_agent():
    agent = UserProxyAgent("user_proxy")
    response = await asyncio.create_task(
        agent.on_messages(
            [TextMessage(content="What is your name? ", source="user")],
            cancellation_token=CancellationToken(),
        )
    )
    assert isinstance(response.chat_message, TextMessage)
    print(f"Your name is {response.chat_message.content}")


RICO Based Prompt Template

In [11]:
from autogen_agentchat.agents import AssistantAgent

meeting_summarizer = AssistantAgent(name="MeetingSummarizer", model_client=model_client,system_message="""  Your are a meeting notes analyst. Your job is to read the meeting notes
                and summarize the key points discussed in the meeting. Don't include any personal opinions or interpretations, 
               just summarize the main points as clearly and concisely as possible.

               Always use the following format for your summary:
               MEETING DATE: [Insert date of the meeting]
                ATTENDEES: [List of attendees]
                KEY POINTS:
                1. [First key point]
                2. [Second key point]   
               ACTION ITEMS:
                1. [First action item] - [owner of the action item] - [due date for the action item]
                2. [Second action item]   - [owner of the action item] - [due date for the action item]
               OPEN QUESTIONS:
               - [questions or None]
               SUMMARY COMPLETE
""")

In [13]:
result = await meeting_summarizer.run(task="""March 25 call — Sarah, James, Priya, and Tom were present.
We decided to push the product launch from April 10 to April 24 because the
payment integration isn't ready. Tom will finish the Stripe integration by April 5.
Priya needs to update the marketing materials to reflect the new date by April 8.
Sarah raised a concern about whether we need a legal review before launch — nobody
had a clear answer on this. James will check with the legal team by March 28.""")
print(result.messages[-1].content)

MEETING DATE: March 25  
ATTENDEES: Sarah, James, Priya, Tom  
KEY POINTS:
1. The product launch has been postponed from April 10 to April 24 due to incomplete payment integration.
2. Tom will complete the Stripe payment integration by April 5.
3. Priya is responsible for updating marketing materials to reflect the new launch date by April 8.
4. Sarah raised a question about whether a legal review is required before the launch, but no clear decision was made.
5. James will consult with the legal team regarding the review by March 28.  

ACTION ITEMS:
1. Tom will finish Stripe payment integration - Tom - April 5  
2. Priya will update marketing materials with the new launch date - Priya - April 8  
3. James will check with the legal team about the review requirement - James - March 28  

OPEN QUESTIONS:
- Is a legal review necessary before the product launch?  

SUMMARY COMPLETE


## Teams - Round Robin

Use case: Content team needs the blog post that passes the quality test and review before publishing it


Flow: Planner -> Writer -> Critic ( round robin , when crtic says APPROVED)

In [23]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console


In [ ]:
content_planner = AssistantAgent(name="ContentPlanner", model_client=model_client,system_message=""" You are a content stratergist.
                                 Your only job : is to create a content for the topic requested by the user.
                                 Do not include any personal opinions or interpretations, just create the content as clearly and concisely as possible.
                                    Always use the following format for your content:
                                 
                                CONTENT TITLE: [Insert title of the content]
                                CONTENT BODY: [Insert body of the content]
                                CONTENT TONE: [Insert tone of the content, e.g., formal, professional, casual, etc.]
                                Key points to consider when creating the content: [3 -4 bullet points]
                                Word count: 150 -200 words
                                MUST INCLUDE: [facts, examples, or angles]     

                                 PLANNER_READY                            
                                 """)

In [16]:
content_writer = AssistantAgent(name="ContentWriter", model_client=model_client,system_message=""" You are a content writer.
                                 Your only job : is to write a blog post based on the content plan created by the content planner agent.
                                 Do not include any new information that is not in the content plan, just write the content based on the content plan provided by the content planner agent.
                                 OUTPUT FORMAT:
                                    BLOG POST TITLE: [Insert title of the blog post]
                                DRAFT:
                                    [Insert draft of the blog post based on the content plan provided by the content planner agent]  

                                DRAFT_COMPLETE                              
                                 """)

In [17]:
content_crtic = AssistantAgent(name="ContentCritic", model_client=model_client,system_message=""" You are a content Quality critic.
                                    Your only job : is to review the content created by the content writer agent and provide feedback on how to improve the content.
                                    Do not rewrite it. just provide constructive feedback on how to improve the content based on the content plan provided by the content planner agent.
                                    EVALUATE:
                                    1. Cover all points from the planner?
                                    2. Is the tone of the content appropriate for the target audience?
                                    3. Is the content clear and concise?
                                    4. Is the word count within the specified range?
                               
                                    OUTPUT FORMAT:
                                    QUALITY SCORE: [/10]
                                    VERDICT: [APPROVED if QUALITY SCORE > 7 else NEED IMPROVEMENT]
                                    ISSUES: [Insert specific issues with the content and suggestions for improvement based on the content plan provided by the content planner agent]

                                    IF APPROVED -> END your response with APPROVED
                                    IF NEED IMPROVEMENT -> END your response with REVISE
                                 
                                    """)

In [22]:
content_team = RoundRobinGroupChat([content_planner, content_writer, content_crtic], name="ContentCreationTeam",
                    termination_condition=TextMentionTermination("APPROVED") | MaxMessageTermination(12) )

In [24]:
result = await content_team.run(task="Create a blog post on the topic of 'The Future of AI in Healthcare'")

In [27]:
for r in result:
    print(r)

('messages', [TextMessage(id='b7719851-f7a4-4623-9454-21e3e6d97225', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 3, 29, 15, 50, 56, 202522, tzinfo=datetime.timezone.utc), content="Create a blog post on the topic of 'The Future of AI in Healthcare'", type='TextMessage'), TextMessage(id='5bfe0451-b15d-483e-a37e-0d0c42454e22', source='ContentPlanner', models_usage=RequestUsage(prompt_tokens=176, completion_tokens=261), metadata={}, created_at=datetime.datetime(2026, 3, 29, 15, 51, 0, 207154, tzinfo=datetime.timezone.utc), content="CONTENT TITLE: The Future of AI in Healthcare\n\nCONTENT BODY:  \nArtificial Intelligence (AI) is poised to revolutionize healthcare in the coming years, transforming patient care, diagnostics, and operational efficiency. AI algorithms are increasingly utilized for early detection of diseases such as cancer, through advanced image analysis and pattern recognition. For example, AI-powered tools like Google's DeepMind have dem

In [28]:
result = await Console(content_team.run_stream(task="Create a blog post on the topic of 'The Future of AI in Healthcare'"))

print(f"STOP REASON:{result.stop_reason}")
print(f"TOTAL MESSAGES:len({result.messages}")

---------- TextMessage (user) ----------
Create a blog post on the topic of 'The Future of AI in Healthcare'
---------- TextMessage (ContentPlanner) ----------
CONTENT TITLE: The Future of AI in Healthcare

CONTENT BODY:  
Artificial Intelligence (AI) is transforming healthcare, offering promising advancements that could redefine patient care and medical operations. AI technologies assist in early disease detection, notably in cancer diagnosis through sophisticated image analysis and pattern recognition. For example, AI systems like IBM Watson are used to support oncologists with treatment recommendations, improving accuracy and speed. Wearable devices powered by AI enable continuous health monitoring, allowing doctors to track vital signs remotely and tailor treatments accordingly. Integration of AI with electronic health records (EHR) enhances data analysis, reducing diagnostic errors and optimizing administrative workflows. Looking forward, innovations such as autonomous robotic sur

## Teams - Selector Group Chat

Use case: Customer Support centre. Different kind ticket needs different agents


Flow: the agent response depends on the what was in the last message 

In [33]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console


In [30]:
ticket_classifier = AssistantAgent(name = "TicketClassifier",model_client=model_client,description="You are a ticket classifier. Your job is to classify the incoming support tickets ",
               system_message=""" You are a support ticket classifier. your only Job ; Classify the incoming ticket.

               Output:
               CATEGORY:[TECHNICAL ISSUE, BILLING ISSUE, ACCOUNT ISSUE, GENERAL INQUIRY]
               URGENCY:[HIGH, MEDIUM, LOW]
               SUMMARY: [A brief summary of the ticket, 1-2 sentences]
               ROUTE TO: [The appropriate team or department to route the ticket to based on the category and urgency]

               CLASSIFIER_READY


            """)

In [32]:
billing_support = AssistantAgent(name = "BillingSupport",model_client=model_client,description="You are a billing support agent. Your job is to handle the billing related support tickets",
                system_message=""" You are a billing support agent. 
                Your only Job : Handle the billing related issues liek disputes, refunds, payment failures, and subscription management.

                Output:
                BILLING RESPONSE: [A clear and concise response to the customer's billing issue, including any necessary steps for resolution or further assistance - don't process any refund without verification]

                ACTION REQUIRED: [ACTION NEED FROM CUSTOMER TO PROVIDE ADDITONAL INFROMATION, NO ACTION REQUIRED]

                RESOLVED

                """)


technical_support = AssistantAgent(name = "TechnicalSupport",model_client=model_client,description="You are a technical support agent. Your job is to handle the technical related support tickets",
                system_message=""" You are a technical support agent. 
                Your only Job : Handle the technical related issues like software bugs, hardware failures, and system errors.

                Output:
                TECHNICAL RESPONSE: [A clear and concise response to the customer's technical issue, including any necessary steps for resolution or further assistance]

                ESCALATION NEEDED:[YES if it requires Engineering team intervention, NO if it can be resolved by the technical support agent]

                RESOLVED

                """)


general_support = AssistantAgent(name = "GeneralSupport",model_client=model_client,description="You are a general support agent. Your job is to handle general support tickets and account related issues",
                system_message=""" You are a general support agent. 
                Your only Job : Handle general support issues and account questions.

                Output:
                GENERAL RESPONSE: [A clear and concise response to the customer's general issue and account questions, including any necessary steps for resolution or further assistance]

                FOLLOW UP NEEDED: [YES if it requires follow up with the customer, NO if it can be resolved in the current response]

                RESOLVED

                """)

In [40]:
support_team = SelectorGroupChat(participants=[billing_support, technical_support, general_support], name="SupportTeam", model_client=model_client, 
                  termination_condition=TextMentionTermination("RESOLVED") | MaxMessageTermination(10),
                   selector_prompt=""" You are a support ticket routing agent.
                     Your job is to classify the incoming support tickets and route them to the appropriate team for resolution.

                     RULES:
                     1. IF THE TICKET HAS NOT BEEN CLASSIFIED YET, FIRST CLASSIFY THE TICKET BASED ON THE CATEGORY AND URGENCY AND THEN ROUTE IT TO THE APPROPRIATE TEAM.
                     2. IF THE TICKET HAS BEEN CLASSIFIED AS BILLING ISSUE, ROUTE IT TO THE BILLING SUPPORT AGENT.
                     3. IF THE TICKET HAS BEEN CLASSIFIED AS TECHNICAL ISSUE, ROUTE IT TO THE TECHNICAL SUPPORT AGENT.
                     4. IF THE TICKET HAS BEEN CLASSIFIED AS ACCOUNT ISSUE OR GENERAL INQUIRY, ROUTE IT TO THE GENERAL SUPPORT AGENT.
                     5. ALWAYS FOLLOW THE OUTPUT FORMAT SPECIFIED BY THE TICKET CLASSIFIER AGENT
                     6. NEVER SELECT TWO AGENT AT SAME TIME. SELECT ONLY ONE AGENT TO HANDLE THE TICKET BASED ON THE CLASSIFICATION AND ROUTING RULES.

                    RETURN ONLY THE NAME OF THE SELECTED AGENT IN YOUR RESPONSE, DO NOT RETURN ANYTHING ELSE OTHER THAN THE NAME OF THE SELECTED AGENT.
                                        
                     
                      
                       """ )

In [39]:
result = await Console(support_team.run_stream(task="""Support ticket from: dev@acme.com
Subject: API returning 429 errors constantly
Message: Our integration has been failing for 2 hours. We're getting
HTTP 429 Too Many Requests on every call even though we're well within
our supposed rate limits. This is blocking our production deployment."""))

print(f"STOP REASON:{result.stop_reason}")
print(f"TOTAL MESSAGES:len({result.messages}")

---------- TextMessage (user) ----------
Support ticket from: dev@acme.com
Subject: API returning 429 errors constantly
Message: Our integration has been failing for 2 hours. We're getting
HTTP 429 Too Many Requests on every call even though we're well within
our supposed rate limits. This is blocking our production deployment.
---------- TextMessage (TechnicalSupport) ----------
TECHNICAL RESPONSE: The 429 Too Many Requests error indicates that the API is throttling your requests, possibly due to exceeding rate limits or a misconfiguration in the rate limiting logic. Please verify that your request rate aligns with the documented limits and ensure that your client is correctly handling rate limit headers. Additionally, check if any recent changes might have affected the rate limiting policy. If the issue persists despite adhering to the limits, please provide your request logs for further analysis.

ESCALATION NEEDED: YES

RESOLVED
STOP REASON:Text 'RESOLVED' mentioned
TOTAL MESSAGES:

In [42]:
result = await Console(support_team.run_stream(task="""Support ticket from: finance@corp.com
Subject: Charged twice this month
Message: We were charged $299 on March 1st and again on March 15th.
We only have one subscription. Please refund the duplicate charge."""))

print(f"STOP REASON:{result.stop_reason}")
print(f"TOTAL MESSAGES:len({result.messages}")

---------- TextMessage (user) ----------
Support ticket from: finance@corp.com
Subject: Charged twice this month
Message: We were charged $299 on March 1st and again on March 15th.
We only have one subscription. Please refund the duplicate charge.
---------- TextMessage (BillingSupport) ----------
BILLING RESPONSE: I understand your concern about the duplicate charges. I have escalated your request to our billing department for further investigation and processing of the refund. Please allow some time for them to review and resolve the issue.

ACTION REQUIRED: No additional action needed from you at this moment. You will be contacted once the refund is processed or if further information is required.

RESOLVED
STOP REASON:Text 'RESOLVED' mentioned
TOTAL MESSAGES:len([TextMessage(id='0ad21a84-bda8-40f9-89e5-c8dd1512012c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 3, 29, 16, 18, 12, 618401, tzinfo=datetime.timezone.utc), content='Support ticket fro

## Research Assistant Team

## RoundRobinGroupChat


## 1. Planner
## 2. Researcher
## 3. Writer
## 4. Critic
## 5. Human Approval

In [34]:
from autogen_agentchat.agents import AssistantAgent,UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [25]:
model_client = OpenAIChatCompletionClient(
    model="gpt-4.1-nano",
    temperature=0.3,
    # api_key="YOUR_API_KEY",
)

In [5]:
## Planner Tool


def score_topic_complexity(topic: str) -> dict:
    """
    Score the complexity of a topic on a scale of 1 to 10, where 1 is very simple and 10 is very complex. 

    Args:
        topic (str): The topic to be scored.
    Returns:
        dict: A dictionary containing the complexity score(LOW/MEDIUM/HIGH) and a description of the topic.
    """

    technical_terms = ["AI", "Machine Learning", "Deep Learning", "Neural Networks", "Natural Language Processing", "Computer Vision","quantum", "neural", "blockchain", "genomic", "cryptographic",
                       "macroeconomic", "geopolitical", "regulatory", "algorithmic"]
    
    has_technical_terms = any(term in topic for term in technical_terms)
    has_comparisons = any(word in topic for word in ["compare", "comparison", "vs", "versus","difference between", "similarities", "contrast"])

    if has_technical_terms and has_comparisons:
        complexity_score = "HIGH"
        description = f"The topic '{topic}' is complex because it involves technical concepts and comparisons."

    elif has_technical_terms:
        complexity_score = "MEDIUM"
        description = f"The topic '{topic}' is moderately complex because it involves technical concepts."

    elif has_comparisons:
        complexity_score = "MEDIUM"
        description = f"The topic '{topic}' is moderately complex because it involves comparisons."
    else:
        complexity_score = "LOW"
        description = f"The topic '{topic}' is simple because it does not involve technical concepts or comparisons."

    return {"score": complexity_score, "description": description}

In [7]:
## Source Credibility Tool

def check_source_credibility(source: str) -> dict:
    """
    Check the credibility of a source based on its domain and provide a credibility score (HIGH/MEDIUM/LOW) along with a brief explanation.

    Args:
        source (str): The source URL to be evaluated.
    Returns:
        dict: A dictionary containing the credibility score and an explanation of the score.
    """

    credible_domains = ["nature.com", "sciencemag.org", "nejm.org", "theatlantic.com", "bbc.com", "reuters.com", "apnews.com"]
    questionable_domains = ["medium.com", "quora.com", "reddit.com", "blogspot.com", "wordpress.com"]

    if any(domain in source for domain in credible_domains):
        credibility_score = "HIGH"
        explanation = f"The source '{source}' is considered highly credible because it is associated with reputable publications known for rigorous editorial standards."

    elif any(domain in source for domain in questionable_domains):
        credibility_score = "LOW"
        explanation = f"The source '{source}' is considered less credible because it is associated with platforms that allow user-generated content without strict editorial oversight."

    else:
        credibility_score = "MEDIUM"
        explanation = f"The source '{source}' has an unknown credibility level because it does not match known credible or questionable domains. Further evaluation may be needed."

    return {"score": credibility_score, "explanation": explanation}

In [17]:
print("Tool unit tests:")
print("score_topic_complexity('AI trends')               :", score_topic_complexity("AI trends"))
print("score_topic_complexity('quantum computing vs blockchain') :", score_topic_complexity("quantum computing impact"))
print()
print("check_source_credibility('apnews.com/economy')     :", check_source_credibility("apnews.com/economy"))
print("check_source_credibility('unknownblog.io/colors')  :", check_source_credibility("unknownblog.io/colors"))

Tool unit tests:
score_topic_complexity('AI trends')               : {'score': 'MEDIUM', 'description': "The topic 'AI trends' is moderately complex because it involves technical concepts."}
score_topic_complexity('quantum computing vs blockchain') : {'score': 'MEDIUM', 'description': "The topic 'quantum computing impact' is moderately complex because it involves technical concepts."}

check_source_credibility('apnews.com/economy')     : {'score': 'HIGH', 'explanation': "The source 'apnews.com/economy' is considered highly credible because it is associated with reputable publications known for rigorous editorial standards."}
check_source_credibility('unknownblog.io/colors')  : {'score': 'MEDIUM', 'explanation': "The source 'unknownblog.io/colors' has an unknown credibility level because it does not match known credible or questionable domains. Further evaluation may be needed."}


In [30]:
# ── Agent 1: Planner ──────────────────────────────────────────────────────────

planner = AssistantAgent(
    name="Planner",
    model_client=OpenAIChatCompletionClient(
    model="gpt-4.1-nano",
    temperature=0.2,
    # api_key="YOUR_API_KEY",
),
    tools=[score_topic_complexity],
    reflect_on_tool_use=True,
    description="Analyses the topic complexity and breaks it into focused research questions.",
    system_message="""
You are a Research Planner.
Your ONLY job: analyse the research topic and create a structured research plan.

Step 1: Call score_topic_complexity(topic) with the exact topic string.
Step 2: Based on the complexity score, decide how many research questions to generate:
        - LOW    → 3 questions
        - MEDIUM → 4 questions
        - HIGH   → 5 questions
Step 3: Output the plan in the format below. Do NOT answer the questions yourself.

Output format:
TOPIC: [exact topic]
COMPLEXITY: [LOW / MEDIUM / HIGH — from tool result]
COMPLEXITY_REASON: [description from tool result]
RESEARCH QUESTIONS:
1. [specific, answerable question]
2. [specific, answerable question]
3. [specific, answerable question]
(4. [if MEDIUM or HIGH])
(5. [if HIGH only])
RESEARCH_SCOPE: [one sentence on what this research will and will not cover]
PLAN_READY
"""
)

In [31]:
# ── Agent 2: Researcher ───────────────────────────────────────────────────────

researcher = AssistantAgent(
    name="Researcher",
    model_client=OpenAIChatCompletionClient(
    model="gpt-4.1-nano",
    temperature=0.3,
    # api_key="YOUR_API_KEY",
),
    tools=[check_source_credibility],
    reflect_on_tool_use=True,
    description="Answers each research question and validates any sources cited.",
    system_message="""
You are a Research Specialist with broad domain knowledge.
Your ONLY job: answer each research question from the Planner's output.

Rules:
- Answer every question from the RESEARCH QUESTIONS list in the plan.
- Whenever you cite a URL or named source, call check_source_credibility(source)
  and include the credibility score next to the citation.
- If a source scores LOW, flag it clearly: [⚠️ LOW CREDIBILITY].
- If a source scores HIGH, mark it: [✅ HIGH CREDIBILITY].
- If a source scores MEDIUM, mark it: [🔍 MEDIUM — verify independently].
- Do NOT write the final report. Do NOT plan. Do NOT evaluate quality.

Output format:
RESEARCH FINDINGS:

Q1: [restate question]
A1: [2-3 sentence factual answer]
SOURCE: [url or "general knowledge"] [credibility badge if a URL was checked]

Q2: [restate question]
A2: [2-3 sentence factual answer]
SOURCE: [url or "general knowledge"] [credibility badge if a URL was checked]

[...continue for all questions]

RESEARCH_COMPLETE
"""
)

In [33]:
# ── Agent 3: Writer — compiles research into a structured report ──
writer = AssistantAgent(
    name="Writer",
    model_client=OpenAIChatCompletionClient(model="gpt-4.1-nano", temperature=0.7),
    description="Writes the final research report from the Researcher's findings.",
    system_message="""
You are a Research Report Writer.
Your ONLY job: compile the research findings into a well-structured, readable report.
If you receive revision notes, address each point specifically before rewriting.

Do NOT conduct additional research. Do NOT evaluate quality. Do NOT add unsupported claims.

Output format:
# [Engaging Report Title]

## Executive Summary
[2-3 sentence overview of key findings]

## Key Findings
[Bullet points — one per research question, with supporting detail]

## Analysis
[2-3 paragraph synthesis connecting the findings]

## Conclusion
[1 paragraph wrapping up the main takeaways]

---
*Research scope: [one sentence on what was and was not covered]*

REPORT_DRAFTED
"""
)

# ── Agent 4: Critic — evaluates quality ───────────────────────────
critic = AssistantAgent(
    name="Critic",
    model_client=OpenAIChatCompletionClient(model="gpt-4.1-nano", temperature=0.2),
    description="Evaluates the report quality and gives APPROVED or specific revision notes.",
    system_message="""
You are a Research Report Critic.
Your ONLY job: evaluate the drafted report against quality standards.
Do NOT rewrite the report. Do NOT conduct research.

Evaluate on 5 criteria (2 points each):
1. Covers all research questions from the plan
2. Claims are factual and not unsupported
3. Report is well-structured with all required sections
4. Analysis connects findings meaningfully
5. Executive Summary accurately reflects the content

Output format:
QUALITY SCORE: [X/10]
CRITERION SCORES:
- Completeness: [0-2]
- Factual accuracy: [0-2]
- Structure: [0-2]
- Analysis depth: [0-2]
- Summary accuracy: [0-2]
VERDICT: [APPROVED if score>=7, else NEEDS_REVISION]
ISSUES: [specific problems, or 'None']
REVISION_NOTES: [exact instructions for Writer, or 'N/A']

If APPROVED → end with: APPROVED
If NEEDS_REVISION → end with: REVISE
If score < 4 (very poor quality) → end with: ESCALATE_TO_HUMAN
"""
)


In [36]:
human_gate = UserProxyAgent(name="HumanReviewer", input_func= input)

In [37]:
researcher_team = RoundRobinGroupChat(participants=[planner, researcher, writer, critic, human_gate], name="ResearchTeam",
                                      termination_condition=TextMentionTermination("APPROVED") | TextMentionTermination("ESCALATE_TO_HUMAN") | MaxMessageTermination(20) )

In [38]:
import time
# ── Research Request 1: Standard topic ───────────────────────────
research_topic_1 = """
Research topic: The impact of remote work on software developer productivity.
Cover the benefits, challenges, tools, and trends since 2020.
Produce a report suitable for a technology leadership audience.
"""

start_time = time.time()

print("=" * 65)
print("RESEARCH REQUEST 1: Remote Work & Developer Productivity")
print("=" * 65)
print()

result1 = await Console(researcher_team.run_stream(task=research_topic_1))

elapsed = time.time() - start_time
print(f"\n{'─'*65}")
print(f"Stop reason  : {result1.stop_reason}")
print(f"Messages     : {len(result1.messages)}")
print(f"Elapsed      : {elapsed:.1f}s")

RESEARCH REQUEST 1: Remote Work & Developer Productivity

---------- TextMessage (user) ----------

Research topic: The impact of remote work on software developer productivity.
Cover the benefits, challenges, tools, and trends since 2020.
Produce a report suitable for a technology leadership audience.

---------- ToolCallRequestEvent (Planner) ----------
[FunctionCall(id='call_INfe3tHRkoJwmB17EllM5kKs', arguments='{"topic": "The impact of remote work on software developer productivity."}', name='score_topic_complexity')]
---------- ToolCallExecutionEvent (Planner) ----------
[FunctionExecutionResult(content='{\'score\': \'LOW\', \'description\': "The topic \'The impact of remote work on software developer productivity.\' is simple because it does not involve technical concepts or comparisons."}', name='score_topic_complexity', call_id='call_INfe3tHRkoJwmB17EllM5kKs', is_error=False)]
---------- TextMessage (Planner) ----------
TOPIC: The impact of remote work on software developer pro